# 00 — Rebuilding World Models Architecture
Goal: Implement the world models architecture to get an intuitive understanding of how it works

In [1]:
import torch
from torch import nn

# Two synthetic images for inspecting tensor shapes.
# These are not training data.
x = torch.rand(2, 3, 64, 64)

conv = nn.Conv2d(
    in_channels=3,
    out_channels=32,
    kernel_size=4,
    stride=2,
)

features = torch.relu(conv(x))

print("Input:", x.shape)
print("Features:", features.shape)

Input: torch.Size([2, 3, 64, 64])
Features: torch.Size([2, 32, 31, 31])


In [2]:
import torch
from torch import nn

encoder = nn.Sequential(
    # [B, 3, 64, 64] → [B, 32, 31, 31]
    nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=0),
    nn.ReLU(),

    # → [B, 64, 14, 14]
    nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=0),
    nn.ReLU(),

    # → [B, 128, 6, 6]
    nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=0),
    nn.ReLU(),

    # → [B, 256, 2, 2]
    nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=0),
    nn.ReLU(),

    # Preserve batch dimension; flatten channels, height, and width.
    # → [B, 1024]
    nn.Flatten(start_dim=1),
)

# Two random RGB images for checking shapes.
x = torch.rand(2, 3, 64, 64)

with torch.no_grad():
    features = x
    print(f"{'Input':<12} {tuple(features.shape)}")

    for layer in encoder:
        features = layer(features)
        print(f"{type(layer).__name__:<12} {tuple(features.shape)}")

assert features.shape == (2, 1024)

Input        (2, 3, 64, 64)
Conv2d       (2, 32, 31, 31)
ReLU         (2, 32, 31, 31)
Conv2d       (2, 64, 14, 14)
ReLU         (2, 64, 14, 14)
Conv2d       (2, 128, 6, 6)
ReLU         (2, 128, 6, 6)
Conv2d       (2, 256, 2, 2)
ReLU         (2, 256, 2, 2)
Flatten      (2, 1024)


In [3]:
# Two separate learned projections of the same CNN features.
mu_head = nn.Linear(1024, 32)
logvar_head = nn.Linear(1024, 32)

features = encoder(x)           # [2, 1024]

mu = mu_head(features)          # [2, 32]
logvar = logvar_head(features)  # [2, 32]

In [4]:
std = torch.exp(0.5 * logvar)
epsilon = torch.randn_like(std)  # Independent standard normal noise.
z = mu + std * epsilon

print("Means:", mu.shape)
print("Log variances:", logvar.shape)
print("Sampled latent:", z.shape)

Means: torch.Size([2, 32])
Log variances: torch.Size([2, 32])
Sampled latent: torch.Size([2, 32])


In [5]:
decoder = nn.Sequential(
    # [B, 32] → [B, 1024]
    nn.Linear(32, 1024),

    # Reshape without changing the values.
    # → [B, 1024, 1, 1]
    nn.Unflatten(dim=1, unflattened_size=(1024, 1, 1)),

    # → [B, 128, 5, 5]
    nn.ConvTranspose2d(1024, 128, kernel_size=5, stride=2),
    nn.ReLU(),

    # → [B, 64, 13, 13]
    nn.ConvTranspose2d(128, 64, kernel_size=5, stride=2),
    nn.ReLU(),

    # → [B, 32, 30, 30]
    nn.ConvTranspose2d(64, 32, kernel_size=6, stride=2),
    nn.ReLU(),

    # → [B, 3, 64, 64]
    nn.ConvTranspose2d(32, 3, kernel_size=6, stride=2),

    # Match our image values, which are scaled to [0, 1].
    nn.Sigmoid(),
)

x_hat = decoder(z)

print("Latent:", z.shape)
print("Reconstruction:", x_hat.shape)

assert x_hat.shape == x.shape

Latent: torch.Size([2, 32])
Reconstruction: torch.Size([2, 3, 64, 64])


In [6]:
# Sum squared pixel errors per image, then average over the batch.
reconstruction_loss = (
    (x_hat - x).square()
    .flatten(start_dim=1)
    .sum(dim=1)
    .mean()
)

In [7]:
# Sum over latent dimensions, then average over images.
kl_loss = 0.5 * (
    mu.square() + logvar.exp() - 1 - logvar
).sum(dim=1).mean()

# Basic VAE objective for learning the mechanism.
beta = 1.0
loss = reconstruction_loss + beta * kl_loss

In [9]:
# Create once, outside the training loop.
parameters = (
    list(encoder.parameters())
    + list(mu_head.parameters())
    + list(logvar_head.parameters())
    + list(decoder.parameters())
)
optimizer = torch.optim.Adam(parameters, lr=1e-4)

In [16]:
# 1. Discard gradients from the previous step.
optimizer.zero_grad()

# 2. Encode images into latent distribution parameters.
features = encoder(x)
mu = mu_head(features)
logvar = logvar_head(features)

# 3. Sample using the reparameterization trick.
std = torch.exp(0.5 * logvar)
epsilon = torch.randn_like(std)
z = mu + std * epsilon

# 4. Reconstruct images.
x_hat = decoder(z)

# 5. Measure reconstruction error and KL divergence.
reconstruction_loss = (
    (x_hat - x).square().flatten(1).sum(1).mean()
)

kl_loss = 0.5 * (
    mu.square() + logvar.exp() - 1 - logvar
).sum(1).mean()

beta = 1.0
loss = reconstruction_loss + beta * kl_loss

# 6. Compute gradients, then update all learned parameters.
loss.backward()
optimizer.step()